# ATINA — imagens das soluções (Kaggle)

**Versão organizada — Stable Diffusion 1.5 + Tesla T4.** Este notebook substitui o fluxo anterior confuso. Mantém o notebook e o CSV privados. Não publiques respostas, pistas, tokens ou SQL.

Segue as células pela ordem. Primeiro faz apenas o teste de 5 imagens. O envio para Supabase fica desligado.

## 1. Preparar ambiente e confirmar GPU
Executa esta célula uma vez por sessão. Ela clona a versão atual do gerador e instala apenas as dependências necessárias.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Ativa Accelerator = GPU nas definições do Kaggle e reinicia a sessão.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')

repo = Path('/kaggle/working/ATINA')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--branch','feature/ai-solution-images','https://github.com/lufime/ATINA.git',str(repo)], check=True)
work = repo / 'tools' / 'ai-solution-images'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(work/'requirements.txt')], check=True)
script = work / 'generate.py'
assert script.is_file(), f'Gerador não encontrado: {script}'
assert 'StableDiffusionPipeline' in script.read_text(encoding='utf-8'), 'O gerador não está na versão Stable Diffusion 1.5.'
print('Preparação concluída. Gerador Stable Diffusion 1.5 confirmado.')

## 2. Encontrar o CSV privado
Adiciona o CSV através de **Input → Add Input**. Não o tornes público.

In [ ]:
import pandas as pd
import json
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
print('CSV encontrados:', len(csv_files))
if not csv_files:
    raise FileNotFoundError('Adiciona o CSV privado em Input → Add Input.')
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
df = pd.read_csv(CSV_PATH, nrows=5)
required = {'challenge_id','category','answer','clues'}
if 'challenges' in df.columns and len(df.columns) == 1:
    raw = pd.read_csv(CSV_PATH, nrows=1).iloc[0]['challenges']
    records = json.loads(raw)
    if not isinstance(records, list) or not records:
        raise ValueError("A coluna 'challenges' não contém uma lista JSON de desafios.")
    df = pd.DataFrame(records)
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltam campos no CSV/JSON: {missing}.')
print('CSV selecionado:', Path(CSV_PATH).name)
print('Formato reconhecido e campos obrigatórios confirmados. (Respostas ocultas.)')

## 3. Credenciais — sem upload nesta fase
O modelo Stable Diffusion 1.5 é público; não é necessário configurar HF_TOKEN se o download público funcionar. O upload para Supabase fica deliberadamente desligado durante o teste.

In [ ]:
UPLOAD_TO_SUPABASE = False
print('Upload Supabase ativo:', UPLOAD_TO_SUPABASE)
print('Nenhuma chave secreta é necessária para o teste local.')

## 4. Teste — gerar apenas 5 imagens
Executa apenas depois de as células 1, 2 e 3 terminarem sem erros. Esta célula volta a localizar os ficheiros após reinícios do Kernel e reutiliza imagens existentes.

In [ ]:
import sys
import subprocess
from pathlib import Path
repo = Path('/kaggle/working/ATINA')
script = repo / 'tools' / 'ai-solution-images' / 'generate.py'
if not script.is_file():
    raise FileNotFoundError('Gerador não encontrado. Executa primeiro a célula 1.')
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
if not csv_files:
    raise FileNotFoundError('CSV privado não encontrado em /kaggle/input.')
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
OUTPUT_DIR = '/kaggle/working/ATINA-generated/images'
cmd = [sys.executable, '-u', str(script), '--manifest', CSV_PATH, '--output', OUTPUT_DIR, '--limit', '5']
print('A iniciar/recomeçar teste. Imagens já existentes serão reutilizadas.', flush=True)
result = subprocess.run(cmd, cwd=str(script.parent), check=False)
generated = sorted(Path(OUTPUT_DIR).glob('*.webp'))
print('\nCódigo de saída:', result.returncode)
print('Ficheiros WebP encontrados:', len(generated))
for p in generated: print(p.name, round(p.stat().st_size/1024), 'KB')

## 5. Rever imagens com as respetivas respostas
Esta célula mostra cada imagem ao lado da resposta correspondente. O conteúdo é privado: não publiques capturas que revelem respostas.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
# Voltar a localizar o CSV e as imagens, mesmo após reinício do Kernel.
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
if not csv_files:
    raise FileNotFoundError('CSV privado não encontrado em /kaggle/input.')
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
OUTPUT_DIR = Path('/kaggle/working/ATINA-generated/images')
raw_df = pd.read_csv(CSV_PATH)
if 'challenges' in raw_df.columns and len(raw_df.columns) == 1:
    records = json.loads(raw_df.iloc[0]['challenges'])
    review_df = pd.DataFrame(records)
else:
    review_df = raw_df
if not {'challenge_id','answer'}.issubset(review_df.columns):
    raise ValueError('O CSV não contém challenge_id e answer para fazer a revisão.')
generated = sorted(OUTPUT_DIR.glob('*.webp')) if OUTPUT_DIR.exists() else []
print('Imagens encontradas:', len(generated))
if not generated:
    print('Não há imagens nesta sessão. Executa primeiro a célula 4.')
for p in generated:
    challenge_id = int(p.stem)
    match = review_df[review_df['challenge_id'].astype(str).str.strip() == str(challenge_id)]
    if match.empty:
        match = review_df[pd.to_numeric(review_df['challenge_id'], errors='coerce') == challenge_id]
    answer = str(match.iloc[0]['answer']) if not match.empty else 'Resposta não encontrada no CSV'
    print(f'Desafio {challenge_id}: {answer}')
    with Image.open(p) as img:
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.title(p.name)
        plt.axis('off')
        plt.show()

## 6. Lote completo — NÃO executar ainda
Só depois de aprovar visualmente o teste. Para já, esta célula fica desativada. A geração dos 1.153 desafios deve ser feita em lotes e com cópias de segurança dos WebP. O upload Supabase será configurado numa etapa separada, com revisão humana.

In [ ]:
# DESATIVADO ATÉ APROVAÇÃO DAS 5 IMAGENS
# cmd = [sys.executable, '-u', str(script), '--manifest', CSV_PATH, '--output', OUTPUT_DIR]
# subprocess.run(cmd, cwd=str(script.parent), check=False)